# WL Kernel + SVM Baseline


In [ ]:
# ── Data loading and filtering ────────────────────────────────────────────────
from pathlib import Path
import networkx as nx
import pandas as pd
import numpy as np

FOLDER = Path('./asnr datasets')
assert FOLDER.exists(), f"Dataset folder not found: {FOLDER}"

asnr_graphs: dict[str, nx.Graph] = {}
for fp in FOLDER.glob('*.graphml'):
    asnr_graphs[fp.stem + '.graphml'] = nx.read_graphml(fp)

info_frame = pd.read_csv(FOLDER / 'Network_summary_masterfile.csv')

selected_classes = ['Mammalia', 'Insecta', 'Aves']
selected_interactions = [
    'physical_contact',
    'social_projection_bipartite',
    'spatial_proximity',
    'grooming',
    'dominance',
    'FE_mating',
]

filtered_df = (
    info_frame[
        info_frame['interaction_type'].isin(selected_interactions) &
        info_frame['class'].isin(selected_classes) &
        (info_frame['nodes'] > 5) &
        pd.notna(info_frame['avg.edge.strength'])
    ]
    .copy()
    .rename(columns={'class': 'network_class'})
    .reset_index(drop=True)
)

interaction_mapping = {c: i for i, c in enumerate(selected_interactions)}
bio_class_mapping   = {c: i for i, c in enumerate(selected_classes)}
filtered_df['interaction_map'] = filtered_df['interaction_type'].map(interaction_mapping)

bio_inv         = {v: k for k, v in bio_class_mapping.items()}
interaction_inv = {v: k for k, v in interaction_mapping.items()}
bio_names       = [bio_inv[i] for i in sorted(bio_inv)]

print(f"Loaded {len(asnr_graphs)} graphs | Filtered: {len(filtered_df)}")


In [ ]:
# ── Dataset construction, split, augmentation ─────────────────────────────────
import copy, random, torch
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data

def build_dataset(graphs_dict, df_subset):
    dataset = []
    for _, row in df_subset.iterrows():
        key = row['Network_ID']
        if key not in graphs_dict:
            continue
        G = graphs_dict[key]
        G = G.to_undirected() if G.is_directed() else G
        node_mapping = {node: i for i, node in enumerate(G.nodes())}
        clustering   = nx.clustering(G)
        x = torch.tensor(
            [[G.degree(n), clustering[n]] for n in G.nodes()], dtype=torch.float)
        edges = []
        for u, v in G.edges():
            edges += [[node_mapping[u], node_mapping[v]],
                      [node_mapping[v], node_mapping[u]]]
        if not edges:
            continue
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        data            = Data(x=x, edge_index=edge_index,
                               y=torch.tensor([row['interaction_map']], dtype=torch.long))
        data.bio_class  = torch.tensor(
            [bio_class_mapping.get(row['network_class'], -1)], dtype=torch.long)
        data.network_id = key
        dataset.append(data)
    return dataset

def augment_node_drop(data, drop_prob=0.05):
    data_aug  = copy.deepcopy(data)
    num_nodes = data_aug.num_nodes
    if num_nodes <= 5:
        return None
    keep_mask = torch.tensor(
        [random.random() > drop_prob for _ in range(num_nodes)], dtype=torch.bool)
    if keep_mask.sum() < 5:
        return None
    data_aug.x = data_aug.x[keep_mask]
    ei        = data_aug.edge_index
    edge_mask = keep_mask[ei[0]] & keep_mask[ei[1]]
    ei        = ei[:, edge_mask]
    old_to_new = torch.full((num_nodes,), -1, dtype=torch.long)
    old_to_new[keep_mask] = torch.arange(keep_mask.sum())
    data_aug.edge_index = old_to_new[ei]
    return data_aug

def oversample_minority(dataset, df, target_col='interaction_map',
                        copies_per_graph=20, drop_prob=0.08, seed=124):
    random.seed(seed); torch.manual_seed(seed)
    counts         = df[target_col].value_counts()
    majority_count = counts.max()
    augmented      = list(dataset)
    for data in dataset:
        label = data.y.item()
        if counts[label] < majority_count * 5:
            for _ in range(copies_per_graph):
                aug = augment_node_drop(data, drop_prob=drop_prob)
                if aug is not None:
                    augmented.append(aug)
                    counts[label] += 1
    return augmented

train_val_df, test_df = train_test_split(
    filtered_df, test_size=0.15,
    stratify=filtered_df['interaction_map'], random_state=42)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.15/0.85,
    stratify=train_val_df['interaction_map'], random_state=42)

train_dataset   = build_dataset(asnr_graphs, train_df)
val_dataset     = build_dataset(asnr_graphs, val_df)
test_dataset    = build_dataset(asnr_graphs, test_df)
augmented_train = oversample_minority(train_dataset, train_df)

print(f"Train: {len(train_dataset)} ({len(augmented_train)} augmented) "
      f"| Val: {len(val_dataset)} | Test: {len(test_dataset)}")


## WL relabelling and feature extraction

In [ ]:
from collections import Counter
import numpy as np
import networkx as nx
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

def wl_relabel(G, node_labels, n_iter):
    """Run n_iter rounds of WL relabelling; return label count histogram."""
    labels = dict(node_labels); counts = Counter()
    for _ in range(n_iter):
        labels = {
            node: hash((labels[node],
                        tuple(sorted(labels[nbr] for nbr in G.neighbors(node)))))
            for node in G.nodes()
        }
        counts.update(labels.values())
    return counts

def get_histograms(nx_graphs, label_dicts, n_iter):
    return [wl_relabel(G, ld, n_iter) for G, ld in zip(nx_graphs, label_dicts)]

def build_wl_vocabulary(histograms):
    """Fit vocabulary on training histograms only."""
    all_keys = sorted(set(k for h in histograms for k in h.keys()))
    return {k: i for i, k in enumerate(all_keys)}

def histograms_to_matrix(histograms, vocab):
    """Convert to fixed-length vectors; ignore labels absent from vocab."""
    X = np.zeros((len(histograms), len(vocab)))
    for i, hist in enumerate(histograms):
        for key, count in hist.items():
            if key in vocab: X[i, vocab[key]] = count
    return X

def dataset_to_nx(dataset):
    nx_graphs, label_dicts = [], []
    for data in dataset:
        G = nx.Graph()
        G.add_nodes_from(range(data.num_nodes))
        G.add_edges_from(data.edge_index.t().numpy())
        degrees = data.x[:,0].numpy().astype(int)
        nx_graphs.append(G)
        label_dicts.append({i: int(degrees[i]) for i in range(data.num_nodes)})
    return nx_graphs, label_dicts

train_nx, train_ld = dataset_to_nx(train_dataset)
val_nx,   val_ld   = dataset_to_nx(val_dataset)
test_nx,  test_ld  = dataset_to_nx(test_dataset)

train_y = [d.y.item() for d in train_dataset]
val_y   = [d.y.item() for d in val_dataset]
test_y  = [d.y.item() for d in test_dataset]
print(f"Converted {len(train_nx)} / {len(val_nx)} / {len(test_nx)} graphs")


## Hyperparameter grid search (validation set)

In [ ]:
print(f"{'n_iter':>8}  {'C':>8}  {'val macro F1':>14}")
best_f1 = 0.0; best_bundle = None

for n_iter in [1, 2, 3]:
    train_hists = get_histograms(train_nx, train_ld, n_iter)
    val_hists   = get_histograms(val_nx,   val_ld,   n_iter)
    vocab       = build_wl_vocabulary(train_hists)   # fit on train only
    X_train     = histograms_to_matrix(train_hists, vocab)
    X_val       = histograms_to_matrix(val_hists,   vocab)
    for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
        svm   = SVC(kernel='rbf', C=C, class_weight='balanced', random_state=42)
        svm.fit(X_train, train_y)
        f1    = f1_score(val_y, svm.predict(X_val), average='macro', zero_division=0)
        print(f"{n_iter:>8}  {C:>8.2f}  {f1:>14.4f}")
        if f1 > best_f1:
            best_f1 = f1; best_bundle = (svm, vocab, n_iter, C)

best_svm, best_vocab, best_iter, best_C = best_bundle
print(f"\nBest: n_iter={best_iter}, C={best_C}  val macro F1: {best_f1:.4f}")


> ⚠️ Run the cell below once only — this is the final test evaluation.

In [ ]:
test_hists = get_histograms(test_nx, test_ld, best_iter)
X_test     = histograms_to_matrix(test_hists, best_vocab)
test_preds = best_svm.predict(X_test)

print("=== WL Kernel + SVM — Test set ===")
print(classification_report(test_y, test_preds,
                             target_names=selected_interactions, zero_division=0))


In [ ]:
cm      = confusion_matrix(test_y, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, fmt, subtitle in zip(
    axes, [cm, cm_norm], ['d','.2f'],
    ['Raw counts','Row-normalised (recall)'],
):
    im = ax.imshow(data, cmap='Greens')
    ax.set_xticks(range(len(selected_interactions)))
    ax.set_yticks(range(len(selected_interactions)))
    ax.set_xticklabels(selected_interactions, rotation=45, ha='right')
    ax.set_yticklabels(selected_interactions)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'WL Kernel + SVM — {subtitle}')
    for i in range(len(selected_interactions)):
        for j in range(len(selected_interactions)):
            ax.text(j, i, format(data[i,j], fmt), ha='center', va='center',
                    fontsize=9,
                    color='white' if data[i,j] > data.max()*0.6 else 'black')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig('confusion_matrix_wl_kernel.pdf', bbox_inches='tight'); plt.show()
